In [166]:
print("Hello")

Hello


In [167]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD

In [168]:
ratings = pd.read_csv('data/ratings.csv')

books = pd.read_csv('data/books.csv')


combined_books_data = pd.merge(ratings,books,on="book_id")

combined_books_data.head()


,book_id,user_id,rating,id,best_book_id,work_id,books_count,isbn,isbn13,authors,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,314,5,27,1,41335427,275,439785960,9.780440e+12,"J.K. Rowling, Mary GrandPré",...,1678823,1785676,27520,7308,21516,136333,459028,1161491,https://images.gr-assets.com/books/1361039191m...,https://images.gr-assets.com/books/1361039191s...
1,1,439,3,27,1,41335427,275,439785960,9.780440e+12,"J.K. Rowling, Mary GrandPré",...,1678823,1785676,27520,7308,21516,136333,459028,1161491,https://images.gr-assets.com/books/1361039191m...,https://images.gr-assets.com/books/1361039191s...
2,1,588,5,27,1,41335427,275,439785960,9.780440e+12,"J.K. Rowling, Mary GrandPré",...,1678823,1785676,27520,7308,21516,136333,459028,1161491,https://images.gr-assets.com/books/1361039191m...,https://images.gr-assets.com/books/1361039191s...
3,1,1169,4,27,1,41335427,275,439785960,9.780440e+12,"J.K. Rowling, Mary GrandPré",...,1678823,1785676,27520,7308,21516,136333,459028,1161491,https://images.gr-assets.com/books/1361039191m...,https://images.gr-assets.com/books/1361039191s...
4,1,1185,4,27,1,41335427,275,439785960,9.780440e+12,"J.K. Rowling, Mary GrandPré",...,1678823,1785676,27520,7308,21516,136333,459028,1161491,https://images.gr-assets.com/books/1361039191m...,https://images.gr-assets.com/books/1361039191s...


In [169]:
rating_utility_matrix = combined_books_data.pivot_table(values = 'rating', index = 'user_id', columns = 'title',fill_value = 0)
rating_utility_matrix.head()

title,'Salem's Lot,"'Tis (Frank McCourt, #2)",1421: The Year China Discovered America,1776,1984,A Bend in the River,A Bend in the Road,A Brief History of Time,A Briefer History of Time,A Case of Need,...,"Women in Love (Brangwen Family, #2)",World War Z: An Oral History of the Zombie War,"World Without End (The Kingsbridge Series, #2)",Wuthering Heights,"Xenocide (Ender's Saga, #3)",Year of Wonders,You Shall Know Our Velocity!,Zen and the Art of Motorcycle Maintenance: An Inquiry Into Values,Zodiac,number9dream
user_id,,,,,,,,,,,,,,,,,,,,,
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [170]:
rating_utility_matrix.shape

(28906, 812)

In [171]:
X = rating_utility_matrix.T

In [172]:
SVD = TruncatedSVD(n_components=30)
transposed_matrix = SVD.fit_transform(X)
transposed_matrix.shape

(812, 30)

In [173]:
corr_matrix = np.corrcoef(transposed_matrix)
corr_matrix.shape

book_titles = rating_utility_matrix.columns
books_list = list(book_titles)


In [174]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

books["content"] = books["authors"] + " " + books["title"]

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(books["content"])

content_sim = cosine_similarity(tfidf_matrix)

indices = pd.Series(books.index, index=books["title"]).drop_duplicates()

idx = indices["1984"]
content_score = content_sim[idx]
    


In [175]:
book_tags = pd.read_csv("data/book_tags.csv")
book_tags.columns

Index(['goodreads_book_id', 'tag_id', 'count'], dtype='object')

In [176]:
tags = pd.read_csv("data/tags.csv")
tags.head()

tags.sample(20)

,tag_id,tag_name
6440,6440,catholocism
28231,28231,soviet-occupation
5499,5499,boss-employee
210,210,100-essential-novels
7332,7332,cis-women
19015,19015,mamet
13926,13926,happy-books
30320,30320,thomas-malory
6558,6558,challenge-ii
11667,11667,feminism-antifeminism


In [177]:
book_tags = book_tags[book_tags["count"] > 100]
bad_tags = [
    "to-read",
    "currently-reading",
    "favorites",
    "owned",
    "default",
    "books-i-own",
    "library"
]

book_tags = book_tags[~book_tags["tag_name"].isin(bad_tags)]

KeyError: 'tag_name'

In [ ]:
book_tags = book_tags.sort_values("count", ascending=False)

top_tags = book_tags.drop_duplicates("goodreads_book_id")

In [ ]:
for tag in top_tags["tag_name"].unique():
    print(tag)

fantasy
classics
young-adult
fiction
historical-fiction
science-fiction
non-fiction
poetry
mystery
horror
graphic-novels
sci-fi
plays
science
philosophy
manga
history
biography
steampunk
chick-lit
urban-fantasy
comics
romance
humor
ya
childrens
book-club
mangá
clàssics
psychology
nonfiction
writing
star-wars
historical-romance
new-adult
dystopian
vampires
travel
paranormal
memoir
stephen-king
short-stories
business
paranormal-romance
graphic-novel
christian
children
zombies
erotica
picture-books
cómics
contemporary
harry-potter
cookbooks
true-crime
parenting
thriller
nicholas-sparks
school
favourites
economics
religion
james-patterson
literature
nora-roberts
dystopia
discworld
angels
self-help
art
spirituality
music
series
house-of-night
christmas
dark
fairy-tales
programming
aliens
agatha-christie
realistic-fiction
drama
feminism
finance
lgbt
religious
sports
india
leadership
china
christian-fiction
jodi-picoult
john-grisham
education
politics
time-travel
kristen-ashley
kindle
contemp

In [ ]:
top_tags.rename(columns={"goodreads_book_id":"book_id"}, inplace=True)

books = pd.merge(
    books,
    top_tags[["book_id","tag_name"]],
    on="book_id",
    how="left"
)

C:\Users\Netra vora\AppData\Local\Temp\ipykernel_9444\2097704132.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_tags.rename(columns={"goodreads_book_id":"book_id"}, inplace=True)


In [ ]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url', 'content', 'tag_name'],
      dtype='object')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

books["content"] = (
    books["authors"].astype(str) + " " +
    books["title"].astype(str) + " " +
    books["tag_name"].astype(str)
)
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(books["content"])

content_sim = cosine_similarity(tfidf_matrix)

indices = pd.Series(books.index, index=books["title"]).drop_duplicates()

idx = indices["1984"]
content_score = content_sim[idx]
    


In [179]:
books.head()

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,content
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,Suzanne Collins The Hunger Games (The Hunger G...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,"J.K. Rowling, Mary GrandPré Harry Potter and t..."
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,"Stephenie Meyer Twilight (Twilight, #1)"
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,Harper Lee To Kill a Mockingbird
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,F. Scott Fitzgerald The Great Gatsby


In [ ]:
def hybrid_recommend(book_name):

    if book_name not in books_list:
        print("Book not found")
        return

    book_index = books_list.index(book_name)
    corr_score = corr_matrix[book_index]

    if book_name not in indices:
        print("Content data not found")
        return

    idx = indices[book_name]

    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    hybrid_scores = []

    for i, title in enumerate(books_list):

        collab = corr_score[i]

        if title in indices:
            content_idx = indices[title]

            if isinstance(content_idx, pd.Series):
                content_idx = content_idx.iloc[0]

            content = content_sim[idx][content_idx]
        else:
            content = 0

        score = (0.4 * collab) + (0.6 * content)
        hybrid_scores.append(score)

    recommended = sorted(
        list(enumerate(hybrid_scores)),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    print("Recommended Books:\n")

    for i in recommended:
        title = books_list[i[0]]
        author = books[books["title"] == title]["authors"].values[0]
        genre = books[books["title"] == title]["tag_name"].values[0]

        print(f"{title} by {author} | Genre: {genre}")

In [ ]:
hybrid_recommend("1984")


Recommended Books:



KeyError: 'tag_name'

In [ ]:
hybrid_recommend("Harry Potter and the Sorcerer's Stone (Harry Potter, #1)")

Recommended Books:

Harry Potter Collection (Harry Potter, #1-6) by 
Harry Potter and the Order of the Phoenix (Harry Potter, #5) by 
Harry Potter and the Half-Blood Prince (Harry Potter, #6) by 
Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5) by 
Harry Potter and the Prisoner of Azkaban (Harry Potter, #3) by 


In [ ]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url', 'content'],
      dtype='object')